# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [1]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 



In [9]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [10]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
*Shirts remaining after Monday's sales:* $50 - 18 = 32$ shirts.

**3. Shipment Received:**
They received a shipment of 30 shirts.
*Total shirts after the shipment:* $32 + 30 = 62$ shirts.

**4. Second Sale:**
They sold half of what they had at this point (half of 62).
*Shirts sold in the second sale:* $62 / 2 = 31$ shirts.

**5. Final Count:**
*Shirts left:* $62 - 31 = 31$ shirts.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: Please provide me with more information so I can answer your question. I do not have access to specific return policies for Acme Corp.

You will need to check:

* **The Acme Corp website:** Look for their "Returns," "Shipping," or "FAQ" sections.
* **Your
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [7]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

# TODO: add two examples before the final input
prompt = f"""Extract name, city, and age as JSON.

Text: {text}
JSON:"""

print(generate(prompt, max_tokens=60))

```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [8]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = ...   # TODO: prompt that asks for only the final answer
cot    = ...   # TODO: prompt that asks for step-by-step reasoning

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=200))

ValidationError: 1 validation error for GenerateRequest
prompt
  Input should be a valid string [type=string_type, input_value=Ellipsis, input_type=ellipsis]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [ ]:
def self_consistent(prompt, k=5):
    # TODO: sample k outputs, extract each final answer, return the most common one
    ...

# print(self_consistent(cot_prompt, k=1))
# print(self_consistent(cot_prompt, k=7))

### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [ ]:
# TODO: extend docs, rebuild vec/doc_vecs, then test
# q = "..."
# print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
# print("With RAG  :", rag_answer(q))

### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.